# W5C1 solution: the network reads the review

The code we wrote together at the front of the room, complete and in order, so
you have the method while you write the comedy version yourself.

**Run every cell from the top. Everything here already works.** The two training
cells take about twelve seconds between them.

The method in one sentence: every word becomes an integer, every integer becomes
a trainable vector, and two different ways of squashing those vectors into one
prediction give two very different answers.

Your job in the lab is the same eight stages on `comedy_reviews.csv`, predicting
`funniness`. **Do not copy this notebook across.** Read the one stage you are
stuck on, close it, and type the comedy version yourself.

## Stage 1. Load the reviews

1200 horror films. `reviews` is a string, `scariness` is a number out of ten.

Notice what is *missing* compared with last class: there are no word lists and
no columns of counts. Nobody is going to tell this model which words matter.

Expect `(1200, 5)`.

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path("../exercise/data")

df = pd.read_csv(DATA / "horror_reviews.csv")
print(df.shape)
print(df.loc[0, "title"], "|", df.loc[0, "scariness"])
print(df.loc[0, "reviews"])

## Stage 2. Every word gets a number

A network cannot multiply the word "terrifying", so every distinct word in the
corpus gets an integer id.

Note what this step is *not*. It is not a judgement about which words are
useful: "the" and "ninety" get ids exactly like "terrifying" does. Sorting out
which ones matter is the model's job now.

`+ 1` is deliberate. **Id 0 is reserved for padding**, and stage 3 is what
spends it.

Expect `89 words`. It is a small vocabulary because the reviews are built from a
small stock of sentence frames; the pipeline is the real thing regardless.

In [ ]:
import re

tokens = [re.findall(r"[a-z]+", r.lower()) for r in df["reviews"]]
vocab = sorted({w for t in tokens for w in t})
stoi = {w: i + 1 for i, w in enumerate(vocab)}   # 0 is left for padding

print(len(vocab), "words")
print(tokens[0][:8])
print([stoi[w] for w in tokens[0][:8]])

## Stage 3. One rectangle of integers

Reviews have different lengths and a tensor is a rectangle, so short reviews get
padded out to the length of the longest one. That is what id 0 was saved for.

**`L - len(t)` is the line worth slowing down on.** It writes each review
flush against the *right* edge, so the padding goes on the **left**.

That sounds arbitrary and it is not. In stage 7 an RNN reads these rows left to
right. Left-padding means it reads the blanks first and finishes on a real word.
Right-padding would have it finish on up to fifty blanks, with the review a
distant memory. If your RNN in the lab comes out no better than your bag of
embeddings, this is the first line to check.

Expect `[1200, 54]`, and the printed row to end in real word ids rather than
zeros.

In [ ]:
import torch

L = max(len(t) for t in tokens)
X = torch.zeros(len(tokens), L, dtype=torch.long)
for i, t in enumerate(tokens):
    X[i, L - len(t):] = torch.tensor([stoi[w] for w in t])   # pad on the LEFT
y = torch.tensor(df["scariness"].values, dtype=torch.float32)

print(X.shape, "|", y.shape, "| longest review", L)
print(X[0, -10:])

## Stage 3b. Look at the data before you model it

Three pictures, no model. Worth more than it looks.

**Left, the thing we are predicting.** Scariness piles up around 5.2 and spreads
either side of it. The width of this histogram is where the 1.972 in the next
stage comes from.

**Middle, why stage 3 looked the way it did.** Reviews run from about 4 tokens
to 54, and the dashed line is the 54 everything was padded out to. Most rows are
mostly padding, which is why `padding_idx=0` matters.

**Right is the one to stare at.** Split the films three ways by what they say
about the single word *terrifying*: never mention it, mention it plainly, or
negate it. Three separated boxes, **5.58** when the word is absent, **6.66**
plain, **4.56** negated.

Read that again, because it is the whole session. Any method that only counts
words puts the middle group and the right-hand group in the same bucket: both
contain "terrifying" exactly once. The gap between those two boxes is about two
points of scariness that a bag of words throws away, and you can see it here
before anything has been trained.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df["length"] = [len(t) for t in tokens]
df["terrifying"] = df["reviews"].str.lower().map(
    lambda t: "negated" if re.search(r"(?:not|never|hardly)(?:\s+\w+)?\s+terrifying", t)
    else "plain" if "terrifying" in t else "absent")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
sns.histplot(df, x="scariness", bins=24, ax=ax[0]).set(title="what we predict")
sns.histplot(df, x="length", bins=24, ax=ax[1]).set(title="review length (tokens)")
ax[1].axvline(L, ls="--", color="C3")
sns.boxplot(df, x="terrifying", y="scariness", order=["absent", "plain", "negated"],
            ax=ax[2]).set(title="the word 'terrifying'", xlabel="")
plt.tight_layout()
plt.show()

print(df.groupby("terrifying")["scariness"].agg(["count", "mean"]).round(2))

## Stage 4. Hold 200 films back, and set the bar

New since last class, and it is the honest half of the method.

Last time the model had four parameters and you could read all four of them out
loud. These models have thousands, and a model with thousands of parameters can
fit 1000 training films without having learned anything that transfers. So "it
fits the data" stops being evidence. The only question left is how it does on
films it has never seen.

Then set the bar *before* training anything. **1.972** is the mean squared error
you get by ignoring the review completely and answering "about 5.2" every time.
Every number below is measured against it.

Expect `1.972`.

In [ ]:
perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(0))
train_i, test_i = perm[:1000], perm[1000:]
Xtr, ytr = X[train_i], y[train_i]
Xte, yte = X[test_i], y[test_i]

print(Xtr.shape, Xte.shape)
print("guess the average every time:  %.3f" % ((yte - ytr.mean()) ** 2).mean())

## Stage 5. The first model: a bag of embeddings

Three lines, one idea each.

- `nn.Embedding` is a lookup table with one row per word id, each row a vector
  of 32 numbers. It starts as noise. **Those numbers are learned like any other
  parameter**, which is the idea to carry out of this stage: the network invents
  its own representation of "terrifying" instead of being handed one.
- `.mean(1)` averages the word vectors into a single vector for the review.
- `nn.Linear` turns that vector into one number.

Now be clear about what `.mean(1)` destroys: **all of the order**. Shuffle the
words of a review and this model returns exactly the same answer. It is a bag of
words with learned vectors in place of counts.

`padding_idx=0` pins row 0 at zero, so the padding adds nothing to the average.

In [ ]:
from torch import nn


class BagOfEmbeddings(nn.Module):
    def __init__(self, V, d=32):
        super().__init__()
        self.emb = nn.Embedding(V, d, padding_idx=0)
        self.out = nn.Linear(d, 1)

    def forward(self, x):
        return self.out(self.emb(x).mean(1)).squeeze(-1)


print(BagOfEmbeddings(len(vocab) + 1))

## Stage 6. The training loop, written once, used twice

The same four beats as last class, **measure, blame, step, clear**, with two
pieces of bookkeeping handed over to PyTorch:

- `torch.optim.Adam` does the `w -= lr * w.grad` you wrote by hand, for every
  parameter in the model, and adapts the step size per parameter as it goes.
- `opt.zero_grad()` is the `.grad.zero_()` you had to remember. Gradients still
  accumulate by default; you just ask for the clear in one call now.

`curve` records two **rows** after each pass, the loss on the training films and
the loss on the held-out ones. Two rows rather than two columns is deliberate:
one row per point, with a column naming the line it belongs to, is the shape
seaborn wants, so the figure in stage 7b needs no reshaping at all. Building the
table in the shape the plot wants is most of what makes plotting easy.

One genuinely new thing: the inner loop takes **32 films at a time** rather than
all 1000. More steps per pass over the data, and each step is cheaper.

Write it once, because it trains *both* of today's models unchanged. That is
what makes the comparison fair: the two models differ by one line and nothing
else, so any difference in the result is about the architecture rather than
about how hard we tried.

Expect the last two rows to read **`train 1.427`** and **`held out 1.416`**, in
a second or two. Better than the 1.972 bar, so the words
really do carry signal and a network found it with nobody writing a word list.
But it is a long way from perfect, and stage 8 is why.

In [ ]:
def train(model, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    curve = []
    for epoch in range(epochs):
        for i in range(0, len(Xtr), 32):
            loss = ((model(Xtr[i:i+32]) - ytr[i:i+32]) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        with torch.no_grad():
            curve.append({"epoch": epoch, "split": "train",
                          "mse": ((model(Xtr) - ytr) ** 2).mean().item()})
            curve.append({"epoch": epoch, "split": "held out",
                          "mse": ((model(Xte) - yte) ** 2).mean().item()})
    return pd.DataFrame(curve)


torch.manual_seed(0)
bag = BagOfEmbeddings(len(vocab) + 1)
bag_curve = train(bag, 150, 0.01)
print(bag_curve.tail(2).round(3).to_string(index=False))

## Stage 7. The second model: read the words in order

Put this class next to the last one. There is exactly one line of difference:

    self.emb(x).mean(1)                  # average the WORDS
    self.rnn(self.emb(x))[0].mean(1)     # average the words IN CONTEXT

`nn.RNN` walks the review left to right carrying a hidden state of 64 numbers.
At each word it mixes that word's vector with everything it has read so far and
emits a new vector. The 32 numbers for "terrifying" go in, and what comes out is
**"terrifying, here, after the words that came before it"**. If one of those
earlier words was "not", the vector that comes out can be a completely different
one.

We still average at the end, so this model is not winning by being bigger or
cleverer at the output. It averages *contextual* vectors instead of
*context-free* ones. That is the only change.

`[0]` takes the outputs at every step. `nn.RNN` also returns its final hidden
state, which we are not using here.

Expect **`train 0.254`** and **`held out 0.635`**, in about twelve seconds. Against 1.416 for the bag, and 1.972
for saying nothing at all.

In [ ]:
class RNNReader(nn.Module):
    def __init__(self, V, d=32, h=64):
        super().__init__()
        self.emb = nn.Embedding(V, d, padding_idx=0)
        self.rnn = nn.RNN(d, h, batch_first=True)
        self.out = nn.Linear(h, 1)

    def forward(self, x):
        return self.out(self.rnn(self.emb(x))[0].mean(1)).squeeze(-1)


torch.manual_seed(0)
rnn = RNNReader(len(vocab) + 1)
rnn_curve = train(rnn, 150, 0.005)
print(rnn_curve.tail(2).round(3).to_string(index=False))

## Stage 7b. Now break it on purpose

The RNN works. Before asking why, spend two minutes making it fail.

**Change exactly one thing: 200 films instead of 1000.** Same architecture, same
loop, same held-out set. Train five times as long, because with a fifth of the
data an epoch is a fifth of the work.

Predict the training loss before you run it. Most people say it will go up,
because there is less data to learn from. It goes **down**, further than before,
to **0.145**, because 200 films are easy to memorize. That is overfitting in one
number.

Note that `Xtr` and `ytr` now point at the small set, so stage 6 and stage 7
would train on 200 films if you re-ran them from here.

Expect **`train 0.145`** and **`held out 2.265`**. Look at the second one
twice: **2.265 is worse than 1.972**, worse than ignoring the review completely
and guessing the average. A model that has nearly memorized its training set can
be worse than useless on anything else.

In [ ]:
Xtr, ytr = X[train_i[:200]], y[train_i[:200]]   # a fifth of the films, same held-out set

torch.manual_seed(0)
starved = RNNReader(len(vocab) + 1)
starved_curve = train(starved, 500, 0.001)
print(starved_curve.tail(2).round(3).to_string(index=False))

## Stage 7c. Three curves, three different things

Three panels, three shapes. The shape is the lesson each time.

**Left, a wall.** Train and held out sit **on top of each other** all the way
down, flatten together at 1.42, and then nothing happens for seventy epochs.
Work out what that rules out:

- Not **overfitting**. That pulls the training curve down and leaves the
  held-out curve behind. These two never separate.
- Not **undertraining**. The curve is flat. It has stopped.

The model already has everything this representation can give it. Ten times the
epochs, ten times the films, a bigger layer: still 1.42.

**Middle, a healthy gap.** The lines separate, to 0.25 against 0.63, and then
both hold steady. It fits the training films better than the held-out ones,
which is normal, and it is not getting worse. This is what a working model looks
like.

**Right, the failure.** Same model, 200 films. The training loss keeps falling,
to 0.145. The held-out curve follows it down, bottoms out around **epoch 200 at
1.557**, then **turns around and climbs** to 2.265, back through the dashed
line. Past epoch 200, every extra step makes the model better at the 200 films
it has seen and worse at everything else.

**Two things worth keeping:**

- *A flat curve and a rising curve are different problems.* The left panel needs
  a better representation, and no amount of extra training or data will fix it.
  The right panel needs *less* training, and that fix is free.
- *The bottom of that orange curve is the model you actually wanted*, and you
  only know where it is because you held 200 films back. That is what a held-out
  set is for, and stopping there has a name: **early stopping**.

In [ ]:
bag_curve["model"] = "bag of embeddings"
rnn_curve["model"] = "rnn reader"
starved_curve["model"] = "rnn reader, 200 films"
curves = pd.concat([bag_curve, rnn_curve, starved_curve], ignore_index=True)

g = sns.relplot(curves, x="epoch", y="mse", hue="split", col="model", kind="line",
                height=3.2, aspect=1.15, facet_kws={"sharex": False})
g.set(ylim=(0, 3.0))
g.set_titles("{col_name}")
g.refline(y=1.972, color="gray", ls="--")

## Stage 8. Why. Two reviews, one bag of words

Read these two before running the cell:

- **A:** the ending is **terrifying**, the score is **not gory**
- **B:** the ending is **not terrifying**, the score is **gory**

A is plainly the better horror film. Now the line that does the work:
`sorted(A.split()) == sorted(B.split())` comes out **True**. Same words, same
counts, one "not" in each. The only difference is *which adjective the "not"
landed on*.

So the bag-of-embeddings model is not just inaccurate on this pair, it is
**blind to it**. It averages the same vectors in both cases and returns
**6.01 twice**, the identical number. That is not a bug and no amount of extra
training would fix it: the information was destroyed by `.mean(1)`, before the
model ever saw it.

The RNN answers **5.25** and **4.44**. The truth is **5.25** and **3.95**.

**A bag of words cannot represent "not".** Every word-counting method in this
course so far has had that hole in it, and this is the first model that does
not.

One question to leave with, because next class starts on it: the RNN only has to
carry "I just read the word *not*" forward a single step here. What happens when
it has to carry something forward forty steps?

In [ ]:
def score(model, text):
    ids = [stoi[w] for w in re.findall(r"[a-z]+", text.lower()) if w in stoi]
    x = torch.zeros(1, L, dtype=torch.long)
    x[0, L - len(ids):] = torch.tensor(ids)
    with torch.no_grad():
        return model(x).item()


tail = " The pacing is boring. The director clearly loves the genre. Ninety minutes, no padding."
A = "The ending is terrifying. The score is not gory." + tail
B = "The ending is not terrifying. The score is gory." + tail

print("same bag of words?", sorted(A.lower().split()) == sorted(B.lower().split()))
for name, text, truth in [("A", A, 5.25), ("B", B, 3.95)]:
    print("%s  truth %.2f | bag %.2f | rnn %.2f"
          % (name, truth, score(bag, text), score(rnn, text)))

## What happens next

The back half of today is the first working session on the **semester project**,
not a second dataset. Open
`project/proposal/CSCI3495_Project_Proposal_Template.docx`, get into your team,
and fill it in. It is due **Wednesday October 7**, and it is one page.

The two things worth having before you leave the room:

- **A question with a question mark in it.** If you cannot write it in one
  sentence, you do not have a project yet, and finding that out today is the
  point of the session.
- **A metric and a baseline.** "We will see if it works" is not a metric. Name
  the number, and name what you compare it against. Today's lab is the shape of
  it: 1.972 was the baseline, 1.416 and 0.635 were the numbers, and none of the
  three means anything without the other two.

### Optional, if you want the repetition

`../exercise/data/comedy_reviews.csv` is 1200 films with a `funniness` score,
built exactly the same way. Redoing all eight stages on it from an empty cell
changes two things: the filename and the column name.

Not assigned and not collected. If you do it, expect every number to come out a
little worse (guessing **2.026**, the bag model **1.518**, the RNN **0.745**),
because comedy ratings are noisier. The ordering of the three is what matters,
and it does not change.